### Graphs 

The rule to remember for your interview

Default to adjacency list

Use:

- dict[int, list[int]] for numbered vertices.

- dict[str, list[str]] for named vertices.

- dict[node, list[node]] or node references when the problem provides objects.

- A grid directly when the input is a matrix of cells.

Don't spend interview time building a full Graph class unless asked to implement graph operations or design a graph API.

Problem #1 

Build a graph from a given edge list

Note: Just pass the flag directed = True and ensure the edge list is appropriate for the directed graph. 

In [10]:
from collections import deque 

# Build a graph from the below edge list 
# n = no. of nodes, edges = no. of edges
def build_graph(n, edges, directed=False):
    graph = {i: [] for i in range(n)}

    for u, v in edges:
        graph[u].append(v)

        if not directed: 
            graph[v].append(u)

    return graph
# Complexity: Time=O(V+E), Space=O(V+E)

# BFS (Using Queue)
def bfs(graph, start):
    queue = deque([start])
    visited = {start}

    while queue:
        node = queue.popleft()
        print (node, end = " - ")

        for neighbor in graph.get(node, []):
            if neighbor not in visited: 
                visited.add(neighbor)
                queue.append(neighbor)

# Complexity with an adjacency list:
# - Time: O(V + E)
# - Auxiliary space: O(V)

# Traversal using iteratibe stack - DFS 
def dfs(graph, start):
    stack = [start]
    visited = set() 

    while stack: 
        node = stack.pop() 
        if node in visited:
            continue 

        visited.add(node)
        print (node, end = " - ")

        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                stack.append(neighbor)


## Usage
edges = [[0, 1], [0, 2], [1, 3], [2, 4]]
graph = build_graph(5, edges)
# basically visits all nodes once
bfs(graph, 2)
print ()
dfs (graph, 2)
print ()
print ()
# print the graph adj list 
for node, edges in graph.items():
    print(node, edges)

2 - 0 - 4 - 1 - 3 - 
2 - 4 - 0 - 1 - 3 - 

0 [1, 2]
1 [0, 3]
2 [0, 4]
3 [1]
4 [2]


DFS explores deeply, while BFS explores by distance in edges.

For large or deep graphs, iterative DFS avoids Python's recursion-depth limitation.

Interview rule: BFS is generally your first choice for shortest paths in unweighted graphs. DFS is often convenient for components, reachability, and cycle detection.


Problem #2 

Number of Islands

Given an m × n grid containing "1" (land) and "0" (water), return the number of islands. Land cells are connected only horizontally or vertically.

How to approach it
    1. Each land cell is a graph vertex.
    2. Neighboring land cells have implicit edges.
    3. Iterate through every cell.
    4. Whenever you encounter unvisited land, you've found a new island.
    5. Run DFS to mark the entire island visited.
    6. Continue searching for the next island.


In [ ]:
def num_islands(grid):
    if not grid:
        return 0

    rows, cols = len(grid), len(grid[0])
    visited = set()
    islands = 0

    directions = [(1,0), (-1,0), (0,1), (0,-1)]

    def dfs(r, c):
        stack = [(r, c)]
        visited.add((r, c))

        while stack:
            x, y = stack.pop()

            for dx, dy in directions:
                nx, ny = x + dx, y + dy

                if (0 <= nx < rows and
                    0 <= ny < cols and
                    grid[nx][ny] == "1" and
                    (nx, ny) not in visited):

                    visited.add((nx, ny))
                    stack.append((nx, ny))

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in visited:
                islands += 1
                dfs(r, c)

    return islands

Time: O(m × n); space: O(m × n) worst case.
Follow-up: What if diagonal connections also count? Add four diagonal directions.

In [ ]:
# Alternative

from collections import deque


def num_islands(grid):
    if not grid or not grid[0]:
        return 0

    rows = len(grid)
    cols = len(grid[0])

    visited = set()
    islands = 0

    directions = [
        (-1, 0),  # Up
        (1, 0),   # Down
        (0, -1),  # Left
        (0, 1)    # Right
    ]

    def explore_island(start_row, start_col):
        queue = deque([(start_row, start_col)])
        visited.add((start_row, start_col))

        while queue:
            row, col = queue.popleft()

            for dr, dc in directions:
                next_row = row + dr
                next_col = col + dc

                # Ignore positions outside the grid.
                if not (0 <= next_row < rows and 0 <= next_col < cols):
                    continue

                # Ignore water.
                if grid[next_row][next_col] != "1":
                    continue

                # Ignore cells already discovered.
                if (next_row, next_col) in visited:
                    continue

                visited.add((next_row, next_col))
                queue.append((next_row, next_col))

    for row in range(rows):
        for col in range(cols):
            if grid[row][col] == "1" and (row, col) not in visited:
                islands += 1
                explore_island(row, col)

    return islands

Problem #3 
Course Schedule
Medium · LC 207

Problem statement
There are numCourses courses and prerequisite pairs [a, b], meaning b must be completed before a. Determine whether all courses can be completed.
For example:
numCourses = 4
prerequisites = [[1,0], [2,0], [3,1], [3,2]]


The real question is "is this directed dependency graph acyclic

Use Kahn's algorithm:

    1. Build a directed adjacency list 
    2. count the incoming edges (indegree) of each vertex 
    3. Start with the vertices whose indegree is zero 
    4. process them, reducing their neighbor's indegrees 
    5. if all the vertices are processed, there is no cycle 



In [11]:
from collections import defaultdict, deque  # defaultdict for adjacency list; deque for BFS queue

def can_finish(numCourses, prerequisites):

    graph = defaultdict(list)  # Adjacency list: each course maps to courses dependent on it
    indegree = [0] * numCourses  # Number of prerequisites each course has

    # Build the directed graph and calculate indegrees
    for course, prereq in prerequisites:
        graph[prereq].append(course)  # Directed edge: prereq -> course
        indegree[course] += 1  # This course has one additional prerequisite

    # Initialize BFS queue with courses having zero prerequisites
    queue = deque(
        i for i in range(numCourses)  # Examine every course from 0 to numCourses - 1
        if indegree[i] == 0  # Include only courses that can be taken immediately
    )

    processed = 0  # Count how many courses have been successfully processed

    # Kahn's Algorithm: BFS-based topological sorting
    while queue:  # Continue as long as courses are available to process

        course = queue.popleft()  # Remove the next course with no remaining prerequisites
        processed += 1  # Mark this course as successfully processed

        # Visit all courses that depend on the completed course
        for next_course in graph[course]:

            indegree[next_course] -= 1  # One prerequisite of next_course is now satisfied

            # If all prerequisites are satisfied, the course is ready
            if indegree[next_course] == 0:
                queue.append(next_course)  # Add it to the BFS queue for processing

    # If all courses were processed, the graph has no directed cycle
    # Otherwise, a cycle prevents one or more courses from being completed
    return processed == numCourses


# Example 1: No cycle — all courses can be completed
print(can_finish(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))  # True

# Example 2: Cycle exists — course 0 needs 1, and course 1 needs 0
print(can_finish(2, [[1, 0], [0, 1]]))  # False

True
False


Problem #4 

Find if Path Exists in Graph
LeetCode 1971 · Easy · BFS / DFS

Given n vertices, undirected edges, a source and a destination, determine whether the destination is reachable.EXAMPLE
n=3, edges=[[0,1],[1,2]], source=0, destination=2 → True


In [ ]:
from collections import deque

def createGraph(n, edges, directed=False):

    graph = {i: [] for i in range(n)}

    for u, v in edges:
        graph[u].append(v) 

        if not directed:
            graph[v].append(u)

    return graph

def bfs(graph, start, destination):
    queue = deque([start])
    visited = {start}

    while queue:
        node = queue.popleft()

        if node == destination:
            print (node, end=".")
            return True

        for neighbor in graph.get(node, []):
            if not neighbor in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return False

def valid_path(n, edges, source, destination) -> bool:
    # A vertex is reachable from itself without taking any edges
    if source == destination:
        return True

    return False

## Usage
n = 3
edges = [[0, 1], [1, 2]]
source = 0 
destination = 2
graph = createGraph(n, edges)
print (graph)

bfs(graph, 1, destination)

{0: [1], 1: [0, 2], 2: [1]}
2.

True

Complexity

| Resource | Complexity |
|---|---:|
| Build adjacency list | O(V + E) time and space |
| BFS traversal | O(V + E) worst-case time |
| Queue and visited set | O(V) space |
| **Overall** | **O(V + E) time and space** |


EdgeCases

| Case | Example | Result |
|---|---|---|
| Source equals destination | `n=1, edges=[], source=0, destination=0` | `True` |
| No connecting edges | `n=2, edges=[], source=0, destination=1` | `False` |
| Direct connection | `edges=[[0,1]], source=0, destination=1` | `True` |
| Disconnected components | `n=4, edges=[[0,1],[2,3]], source=0, destination=3` | `False` |
| Cycle | `edges=[[0,1],[1,2],[2,0]], source=0, destination=2` | `True`; visited prevents repetition |
| Duplicate edges or self-loops | If allowed | Handled by the visited check |


Can we use DFS ?

Yes, replace deque with a stack (list) and popleft() with pop() 

Problem #5 

Number of Islands
LeetCode 200 · Medium · Grid DFS

Count groups of horizontally or vertically connected land cells in a binary grid 

EXAMPLE
[[1,1,0],[0,1,0],[0,0,1]] → 2


In [23]:

from collections import deque


def explore_land(grid, row, col, visited):
    rows, cols = len(grid), len(grid[0])

    queue = deque([(row, col)])
    visited.add((row, col))

    while queue:
        row, col = queue.popleft()

        # Check up, down, left, and right.
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = row + dr, col + dc

            if not (0 <= nr < rows and 0 <= nc < cols):
                continue  # Outside the grid

            if grid[nr][nc] == 1 and (nr, nc) not in visited:
                visited.add((nr, nc))  # Mark before enqueueing
                queue.append((nr, nc))


def count_land_groups(grid):
    if not grid or not grid[0]:
        return 0

    visited = set()
    groups = 0

    for row in range(len(grid)):
        for col in range(len(grid[0])):
            if grid[row][col] == 1 and (row, col) not in visited:
                groups += 1
                explore_land(grid, row, col, visited)

    return groups

# ==========
grid = [
    [1, 1, 0],
    [0, 1, 0],
    [0, 0, 1]
]

print(count_land_groups(grid))  # 2

2


Complexity: O(rows × cols) time and O(rows × cols) worst-case auxiliary space. The input grid remains unchanged.

The important distinction: increment the count when starting a new BFS, not for every land cell visited.

Problem #6

Rotting Oranges
LeetCode 994 · Medium · Multi-source BFS

A grid contains empty cells, fresh oranges and rotten oranges. Each minute, rotten oranges infect adjacent fresh oranges. Return the minimum time until no fresh oranges remain, or -1 if impossible.

EXAMPLE
[[2,1,1],[1,1,0],[0,1,1]] → 4

Approach: 

Assume:
1. 
    0 = empty cell 
    1 = fresh orange 
    2 = rotten orange

2. Only up, down, left and right neighbours are infected. 
    directtions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

We're gonna use BFS: Starting with all rotten oranges in the queue and each BFS level represents one minute of simultaneous infection. 

With the above assumptions only up, down, left and right oranges are infected (i.e. spread of infection)

the Steps.. 
1. Scan the grid and enqueue every rotten orange 
2. count the fresh oranges 
3. Process the queue one level at a time 
4. Each level infect the next layer of fresh oranges
5. If fresh oranges remain after spreadin stops, return -1 

[
    [2,1,1],
    [1,1,0],
    [0,1,1]
]
            Newly rotten cells      Remaining Fresh oranges
Initially at (0, 0)                         6
Minute 1    (0, 1) (1, 0)                   4
Minute 2    (0, 2) (1, 1)                   2
Minute 3    (2, 1)                          1
Minute 4    (2, 2)                          0

Answer 4 minutes


In [ ]:
from collections import deque  # Queue with efficient removal from the front.


def oranges_rotting(grid):  # Return minutes needed, or -1 if infection cannot reach all fresh oranges.
    if not grid or not grid[0]:  # Handle an empty grid or a grid with no columns.
        return 0  # No fresh oranges means no waiting time.

    rows, cols = len(grid), len(grid[0])  # Get the grid dimensions.
    queue = deque()  # Store rotten oranges whose neighbours need to be checked.
    fresh = 0  # Count fresh oranges still remaining.

    for row in range(rows):  # Visit each row.
        for col in range(cols):  # Visit each column in that row.
            if grid[row][col] == 2:  # This orange is initially rotten.
                queue.append((row, col))  # Add it as a starting point for simultaneous BFS.
            elif grid[row][col] == 1:  # This orange is initially fresh.
                fresh += 1  # Increase the fresh-orange count.

    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Moves: up, down, left, right.
    minutes = 0  # Track completed rounds of infection.

    while queue and fresh > 0:  # Continue while infection can spread and fresh oranges remain.
        level_size = len(queue)  # Freeze the current layer size; new additions wait for the next minute.

        for _ in range(level_size):  # Process only oranges belonging to this minute's starting layer.
            row, col = queue.popleft()  # Remove the next rotten orange to examine.

            for dr, dc in directions:  # Check each of its four neighbouring positions.
                nr, nc = row + dr, col + dc  # Calculate the neighbour's row and column.

                if not (0 <= nr < rows and 0 <= nc < cols):  # Check whether the neighbour is outside the grid.
                    continue  # Skip this direction and try the next one.

                if grid[nr][nc] != 1:  # Check whether this cell is empty or already rotten.
                    continue  # Only fresh oranges can become newly infected.

                grid[nr][nc] = 2  # Turn it rotten immediately; also prevents enqueueing it twice.
                fresh -= 1  # One fewer fresh orange remains.
                queue.append((nr, nc))  # Schedule this orange to spread infection in the next round.

        minutes += 1  # One round of simultaneous spreading has finished.

    return minutes if fresh == 0 else -1  # Return elapsed time, or -1 if fresh oranges remain unreachable.


grid = [  # Example input: 0 = empty, 1 = fresh, 2 = rotten.
    [2, 1, 1],  # The initial rotten orange is at (0, 0).
    [1, 1, 0],  # Empty cells cannot transmit infection.
    [0, 1, 1]   # The orange at (2, 2) becomes rotten last.
]

print(oranges_rotting(grid))  # Prints 4; the function modifies grid in place.

Complexity 

| Resource | Complexity |
|---|---:|
| Time | **O(rows × cols)** |
| Auxiliary space | **O(rows × cols)** worst case for the queue |


Ege Cases 
| Grid | Result | Reason |
|---|---:|---|
| `[]` | `0` | No fresh oranges |
| `[[0]]` | `0` | Empty cell |
| `[[2]]` | `0` | Already rotten |
| `[[1]]` | `-1` | No source of infection |
| `[[2, 1]]` | `1` | One adjacent fresh orange |
| `[[2, 0, 1]]` | `-1` | Empty cell blocks infection |
| `[[2, 0], [0, 1]]` | `-1` | Diagonal infection is not allowed |
| `[[2, 1, 1, 1, 2]]` | `2` | Both ends spread simultaneously |



Problem #7

Course Schedule
LeetCode 207 · Medium · Cycle detection

Given courses and prerequisite dependencies, determine whether all courses can be completed.

EXAMPLE
numCourses=2, prerequisites=[[1,0],[0,1]] → False

All the courses can be completed if their prerequisite dependencies contain no directed cycles.

Assumptions:

Assume each pair [course, prerequisite] means 

[1, 0] => Complete course 0 before course #1

So, with the given example:
prerequisites=[[1,0],[0,1]]

- Course 1 required course 0 
- Course 0 requires course 1 

Neither can be started, so the answer is False

So: Now what is the starting point.
Any course with ZERO unfinished prerequisites

So, the steps would be to go with ... 

1. Put all such courses into a queue 
2. Complete a course from the queue 
3. Reduce the unfinished-prerequisite count of courses depending on it 
4. If a dependednt course's content becomes Zero, enqueue it 
5. Check whether you completed every course. 

So.. we need two structures here ..
dependents[p] - courses unlocked by completing prerequisite p 
indegree[c].   - number of prerequisite course c still needs 


In [25]:
from collections import deque  # Queue for courses ready to take.


def can_finish(numCourses, prerequisites):
    dependents = [[] for _ in range(numCourses)]  # One dependent-course list per course.
    indegree = [0] * numCourses  # Initially, every course has zero known prerequisites.

    for course, prerequisite in prerequisites:  # Read each dependency.
        dependents[prerequisite].append(course)  # Add directed edge: prerequisite → course.
        indegree[course] += 1  # Count one prerequisite for this course.

    queue = deque()  # Courses whose prerequisites have all been satisfied.

    for course in range(numCourses):  # Include all courses, even those absent from the edge list.
        if indegree[course] == 0:  # This course has no prerequisites.
            queue.append(course)  # It can be taken immediately.

    completed = 0  # Count how many courses we successfully process.

    while queue:  # Continue while at least one course is ready.
        course = queue.popleft()  # Choose a ready course.
        completed += 1  # Treat this course as completed.

        for dependent in dependents[course]:  # Examine courses requiring the completed course.
            indegree[dependent] -= 1  # One of their prerequisites is now satisfied.

            if indegree[dependent] == 0:  # All prerequisites for this course are satisfied.
                queue.append(dependent)  # This course is now ready to take.

    return completed == numCourses  # False means a cycle prevented full completion.

## Usage 
numCourses = 2
prerequisites=[[1,0],[0,1]]

print(can_finish(2, [[1, 0], [0, 1]]))  # False
print(can_finish(2, [[1, 0]]))          # True



False
True


Trace your example
After building the graph:
dependents = [
    [1],  # Completing 0 would unlock 1.
    [0]   # Completing 1 would unlock 0.
]

indegree = [1, 1]

Both courses have one pre-requisite, so: 
queue = deque([])  # No course can start.
completed = 0

The loop never runs: 
completed == numCourses
0 == 2  # False


A successful example would be: 

numCourses = 4
prerequisites = [[1, 0], [2, 0], [3, 1], [3, 2]]

Why no visited set?
A course enters the queue when its unfinished-prerequisite count reaches zero. With this algorithm, that happens once per course.

Unlike ordinary BFS, discovering a neighbour does not make it ready. Every prerequisite must be satisfied first.

Complexity

Let V be the number of courses and E the number of prerequisite pairs.
- Time: O(V + E)—process every course and dependency once.
- Auxiliary space: O(V + E)—adjacency lists, counts, and queue.

Edge cases and follow-ups

| Case or question | Answer |
|---|---|
| No prerequisites | `True`; all courses are initially ready |
| Course requires itself: `[[0, 0]]` | `False`; self-cycle |
| Some courses are disconnected | Fine; include every course in the initial scan |
| One component contains a cycle | `False`, even if other courses can finish |
| Return a valid completion order? | Append each removed course to a result list; return it only if all courses complete |
| Can DFS solve it? | Yes, using states: unvisited, currently exploring, finished |
| Why isn’t a simple visited set enough for DFS cycle detection? | Revisiting a finished node is valid; reaching a node still on the current DFS path indicates a cycle |

This also models task dependencies in a workflow: a task becomes ready only after all its prerequisites are complete.

Problem #8 

Course Schedule II
LeetCode 210 · Medium · Topological sort

Return a valid ordering of all courses respecting prerequisites, or an empty list if impossible.

EXAMPLE
numCourses=2, prerequisites=[[1,0]] → [0,1]


Problem #9 

Number of Connected Components
LeetCode 323 · Medium · Components

Given n nodes and undirected edges, count how many connected groups exist, including isolated vertices.EXAMPLE
n=5, edges=[[0,1],[1,2],[3,4]] → 2

Problem #10 

Clone Graph
LeetCode 133 · Medium · DFS/BFS + hashmap

Given a node in a connected undirected graph, create a deep copy of all reachable nodes and their connections.EXAMPLE
1↔2, 2↔3 → independent copied graph with the same structure

Problem #11 

Word Ladder
LeetCode 127 · Hard · Shortest path BFS

Transform beginWord to endWord, changing one letter per step, using only valid intermediate dictionary words. Return the shortest transformation sequence length.EXAMPLE
hit → hot → dot → dog → cog → 5 words